# Explainable Chronos — End-to-End Demo

This notebook shows **Extension 1** and **Extension 2** working together.

**Flow**
1. Generate a Chronos-2 forecast with covariates (shared forecast provider)
2. Extension 1 — explain the forecast: attribution, trajectory, NLI-scored verbalization
3. Extension 2 — query the forecast in natural language: `remove_covariate`, `scale_covariate`, `change_horizon`, `confidence_query`

**Scenario**: 200-step synthetic sales series with three covariates — `marketing_spend`, `promo_discount`, `competitor_sales`.

In [ ]:
!git clone https://github.com/Voyagers-time-series-forecasting/explainable-chronos.git
%cd explainable-chronos
!pip install --upgrade pip setuptools wheel
!pip install -e .

In [ ]:
import sys, pathlib, warnings, textwrap
warnings.filterwarnings("ignore")

repo_root = pathlib.Path().resolve()
if not (repo_root / "extension_1").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch

# Extension 1
from extension_1.pipeline import VerbalizationPipeline
from extension_1.config import PipelineConfig
from extension_1.evaluation.scoring import NLIConsistencyScorer
from extension_1.verbalization.template import TemplateVerbalizer
from extension_1.attribution.types import CovariateSet

# Extension 2
from extension_2.dialogue import DialogueSystem

# Shared
from shared.forecast_provider import ChronosForecastProvider

print("Imports OK")

## 1 — Synthetic Data

In [ ]:
N, HORIZON = 200, 14
rng = np.random.RandomState(42)
t   = np.arange(N + HORIZON, dtype=float)

target   = 300 + 1.2 * t + rng.randn(N + HORIZON) * 10
mkt      = 80  + 0.8 * t + rng.randn(N + HORIZON) * 8
promo    = 20  + 0.1 * t + rng.randn(N + HORIZON) * 4
comp     = 350            + rng.randn(N + HORIZON) * 20

history  = target[:N]
actuals  = target[N:]
past_cov = {"marketing_spend": mkt[:N],  "promo_discount": promo[:N],  "competitor_sales": comp[:N]}
fut_cov  = {"marketing_spend": mkt[N:],  "promo_discount": promo[N:],  "competitor_sales": comp[N:]}

COV_NAMES  = list(past_cov.keys())
COLORS     = {"marketing_spend": "#e67e22", "promo_discount": "#27ae60", "competitor_sales": "#8e44ad"}

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(np.arange(N), history, color="#2c3e50", lw=1.5, label="History")
ax.plot(np.arange(N, N+HORIZON), actuals, color="#e74c3c", lw=1.5, ls="--", label="Actuals (hold-out)")
ax.axvline(N, color="#aaa", lw=0.8, ls=":")
ax.set_title("Synthetic Sales — 200 steps history + 14-step horizon", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2 — Extension 1: Forecast + Explanation

In [ ]:
device   = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

provider   = ChronosForecastProvider(device=device, enable_attention=True)
verbalizer = TemplateVerbalizer(seed=42)
scorer     = NLIConsistencyScorer()
config     = PipelineConfig(horizon=HORIZON, downside_factor=0.92, upside_factor=1.08)

pipeline = VerbalizationPipeline(
    forecast_provider=provider,
    verbalizer=verbalizer,
    scorer=scorer,
    config=config,
)

cov_set     = CovariateSet.from_dict(past_cov)
fut_cov_set = CovariateSet.from_dict(fut_cov)

result = pipeline.run(history, horizon=HORIZON, covariates=cov_set, future_covariates=fut_cov_set)
print(f"NLI: {result.consistency_report.overall_score:.3f}  "
      f"({'PASS' if result.consistency_report.is_consistent else 'FAIL'})")

In [ ]:
# Visualise Extension 1 output
p10 = np.asarray(result.forecast["p10"])
p50 = np.asarray(result.forecast["p50"])
p90 = np.asarray(result.forecast["p90"])
tail = 60
h_steps = np.arange(-tail, 0)
f_steps = np.arange(HORIZON)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Forecast panel
ax = axes[0]
ax.plot(h_steps, history[-tail:], color="#2c3e50", lw=1.5, label="History")
ax.fill_between(f_steps, p10, p90, alpha=0.25, color="#2980b9")
ax.plot(f_steps, p50, color="#2980b9", lw=2.2, label="P50")
ax.plot(f_steps, actuals, color="#e74c3c", ls="--", lw=1.5, label="Actual")
ax.axvline(0, color="#aaa", lw=0.8, ls=":")
f = result.features
ax.set_title(f"Trend: {f.trend_magnitude} {f.trend_direction} | Uncertainty: {f.uncertainty_level}", fontsize=9)
ax.legend(fontsize=8); ax.tick_params(labelsize=8)

# Attribution panel
ax = axes[1]
attrs  = result.attribution.attributions[:result.attribution.top_k]
names  = [a.name.replace("_", " ").title() for a in attrs]
values = [a.relative_impact_pct for a in attrs]
colors = [COLORS.get(a.name, "#95a5a6") for a in attrs]
y = np.arange(len(names))
bars = ax.barh(y, values, color=colors, alpha=0.85)
for bar, v in zip(bars, values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f"{v:.1f}%", va="center", fontsize=8)
ax.set_yticks(y); ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("Relative impact (%)", fontsize=8)
ax.set_title("Covariate Attribution — Attention Rollout", fontsize=9)

plt.suptitle("Extension 1 Output", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

print("\nGenerated summary:")
print(textwrap.fill(result.verbalization.summary, width=120))

## 3 — Extension 2: Natural-Language Dialogue

The `DialogueSystem` wraps the same forecast provider. Each `query()` call parses the user's intent, modifies covariates/horizon accordingly, re-runs Chronos-2, and returns a natural-language answer with the updated forecast.

In [ ]:
from extension_1.attribution.types import CovariateSet

dialogue = DialogueSystem(
    history=history,
    covariates=cov_set,
    horizon=HORIZON,
    seed=42,
)
print("DialogueSystem ready.")

In [ ]:
# Helper: send a query, print the answer, and plot the updated P50 vs baseline P50

def ask(system: DialogueSystem, query: str, baseline_p50=p50) -> None:
    print(f"\n>>> {query}")
    resp = system.query(query)
    print(f"Intent   : {resp.intent.intent_type}  (confidence={resp.intent.confidence})")
    print(f"Covariate: {resp.intent.target_covariate}")
    print(f"NLI score: {resp.consistency_score:.3f}  ({'PASS' if resp.is_consistent else 'FAIL'})")
    print()
    print(textwrap.fill(resp.answer, width=110))

    if resp.delta_p50 is not None and not np.allclose(resp.delta_p50, 0):
        new_p50 = baseline_p50 + resp.delta_p50[:HORIZON]
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(f_steps, baseline_p50, color="#2980b9", lw=1.8, ls="--", label="Baseline P50")
        ax.plot(f_steps, new_p50,      color="#e74c3c", lw=1.8,          label="Updated P50")
        ax.fill_between(f_steps, baseline_p50, new_p50, alpha=0.15, color="#e74c3c")
        ax.set_title(f"P50 delta after: \"{query}\"", fontsize=9)
        ax.legend(fontsize=8); ax.tick_params(labelsize=8)
        plt.tight_layout(); plt.show()

In [ ]:
# Query 1: Confidence / uncertainty
ask(dialogue, "How confident are you in this forecast?")

In [ ]:
# Query 2: Remove a covariate (counterfactual ablation)
ask(dialogue, "What would happen if we removed the marketing spend covariate?")

In [ ]:
# Query 3: Scale a covariate
ask(dialogue, "What if marketing spend doubled?")

In [ ]:
# Query 4: Change the forecast horizon
ask(dialogue, "Show me the next 7 days instead.")

## 4 — Multi-Turn: State Persistence

State modifications (covariate removal, scaling) persist across turns — the system does not reset between queries.

In [ ]:
# Fresh system for multi-turn demo
mt = DialogueSystem(history=history, covariates=cov_set, horizon=HORIZON, seed=42)

print("Turn 1:")
r1 = mt.query("Remove marketing_spend from the forecast.")
print(f"  Intent: {r1.intent.intent_type} | modified={r1.modification.modified}")
print(f"  {textwrap.fill(r1.answer, 100)}")

print("\nTurn 2 (marketing_spend should still be zeroed):")
r2 = mt.query("How confident are you in this forecast?")
print(f"  Intent: {r2.intent.intent_type}")
print(f"  NLI: {r2.consistency_score:.3f}")
print(f"  {textwrap.fill(r2.answer, 100)}")